# Evaluation CNN per FER-2013

Questo notebook carica il modello salvato da `Training.ipynb` e lo valuta su `data/original/test` con messaggi di avanzamento.


In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime
from contextlib import contextmanager
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


def log_step(message):
    now = datetime.now().strftime("%H:%M:%S")
    print(f"[{now}] {message}", flush=True)


@contextmanager
def timed_step(name):
    start_time = perf_counter()
    log_step(f"INIZIO - {name}")
    try:
        yield
    finally:
        elapsed = perf_counter() - start_time
        log_step(f"FINE - {name} ({elapsed:.1f}s)")


SEED = 42
IMG_HEIGHT = 48
IMG_WIDTH = 48
BATCH_SIZE = 32

log_step(f"TensorFlow: {tf.__version__}")
log_step(f"GPU disponibili: {tf.config.list_physical_devices('GPU')}")


In [ ]:
with timed_step("Configurazione percorsi evaluation"):
    current_dir = Path.cwd()
    project_root = current_dir if (current_dir / "data").exists() else current_dir.parent

    model_path = project_root / "models" / "best_cnn.keras"
    test_dir = project_root / "data" / "original" / "test"

    if not model_path.exists():
        raise FileNotFoundError(f"Modello non trovato: {model_path}. Esegui prima notebooks/Training.ipynb.")
    if not test_dir.exists():
        raise FileNotFoundError(f"Cartella test non trovata: {test_dir}")

    log_step(f"Project root: {project_root}")
    log_step(f"Model path: {model_path}")
    log_step(f"Test directory: {test_dir}")


In [ ]:
with timed_step("Caricamento modello e test generator"):
    model = load_model(model_path)

    test_datagen = ImageDataGenerator(rescale=1./255)
    test_generator = test_datagen.flow_from_directory(
        test_dir,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        color_mode="grayscale",
        class_mode="categorical",
        shuffle=False
    )

    emotion_labels = {v: k for k, v in test_generator.class_indices.items()}

    log_step(f"Test samples: {test_generator.samples}")
    log_step(f"Batch di test: {len(test_generator)}")
    log_step(f"Classi: {test_generator.class_indices}")


In [ ]:
with timed_step("Valutazione modello salvato"):
    log_step("Eseguo model.evaluate")
    test_loss, test_accuracy, test_auc = model.evaluate(test_generator, verbose=1)

    log_step(f"Test loss: {test_loss:.4f}")
    log_step(f"Test accuracy: {test_accuracy:.4f}")
    log_step(f"Test AUC: {test_auc:.4f}")


In [ ]:
with timed_step("Predizioni e report"):
    log_step("Calcolo predizioni sul test set")
    y_true = test_generator.classes
    y_pred_proba = model.predict(test_generator, verbose=1)
    y_pred = np.argmax(y_pred_proba, axis=1)

    target_names = [emotion_labels[i] for i in range(len(emotion_labels))]

    log_step("Classification report")
    print(classification_report(y_true, y_pred, target_names=target_names))

    log_step("Confusion matrix")
    cm = confusion_matrix(y_true, y_pred)
    print(cm)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    fig, ax = plt.subplots(figsize=(9, 9))
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45)
    plt.title("Confusion matrix - test set")
    plt.tight_layout()
    plt.show()
